# Plant Identifier — Model Training (Week 1)

This notebook trains the leaf classifier used by the Plant Identifier API.
It is designed to work with **Runtime → Run all** on Google Colab (free T4 GPU).

**What it does**
1. Downloads PlantVillage (lab photos), PlantDoc (field photos) and a rice leaf disease dataset.
2. Maps every source folder to one label format: `Plant___Condition` (e.g. `Tomato___Early_blight`).
3. Removes exact and near-duplicate images, then splits train/val/test **without leakage**.
4. **Stage A** — fine-tunes MobileNetV3 on PlantVillage only (the "lab-only" baseline).
5. **Stage B** — continues from Stage A with PlantDoc + rice field photos and field-style augmentation.
6. Evaluates both models separately on the lab test set and the field test set
   at two levels: **disease** (41 classes) and **plant** (15 crops, probabilities summed per crop).
7. Exports Stage B to ONNX, checks ONNX Runtime output matches PyTorch, and writes `labels.json`
   (labels, plants, preprocessing, and which plants have a reliable leaf-condition diagnosis).

**Before you press Run all**
- `Runtime → Change runtime type → T4 GPU`. Nothing else needs editing.
- Rice is downloaded from Kaggle only if the Colab secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` exist
  (🔑 icon in the left sidebar, notebook access on; token from kaggle.com → Settings → API).
  Without them the notebook still runs, just without rice. Never paste the key into a cell.
- Colab asks once for **Google Drive** access: allow it. Checkpoints are saved to
  `MyDrive/plant-identifier/run-<id>/` after every epoch. If the session disconnects (for example when the
  free GPU quota runs out), open the notebook again and press **Run all**: finished stages are skipped and
  an interrupted stage resumes from its last epoch. Datasets are downloaded again (about 10 minutes).

Expected run time on a T4: roughly 60–90 minutes.

**License:** the model is trained on PlantVillage (CC BY-SA 3.0), so the model and its outputs are
released under CC BY-SA 3.0. PlantDoc is CC BY 4.0; the rice dataset is CC0 1.0.

## 1. Setup

In [ ]:
!nvidia-smi -L || echo "No GPU found: switch the runtime to T4 GPU"
%pip install -q onnx onnxruntime onnxscript

In [ ]:
import io, os, re, json, copy, math, time, random, shutil, hashlib, subprocess
from collections import Counter, defaultdict
from datetime import datetime, timezone
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile, ImageOps
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ---------------- Configuration ----------------
SEED = 42

# Rice Disease Dataset (Shrestha et al., 2021, CC0 1.0), DOI 10.34740/KAGGLE/DSV/2481060.
# Downloaded only if the Kaggle secrets exist; otherwise the notebook continues without rice.
RICE_KAGGLE_SLUG = "nischallal/rice-disease-dataset"
INCLUDE_RICE = True

WORK_DIR = Path("/content/work")
RAW_DIR = WORK_DIR / "raw"
CACHE_DIR = WORK_DIR / "cache"
ARTIFACT_DIR = Path("/content/artifacts")

PV_REPO_DIR = RAW_DIR / "PlantVillage-Dataset"
PV_COLOR_DIR = PV_REPO_DIR / "raw" / "color"
PLANTDOC_DIR = RAW_DIR / "PlantDoc-Dataset"
RICE_DIR = RAW_DIR / "rice"

CACHE_SHORT_SIDE = 320          # images are resized once (shorter side) to speed up training
IMG_SIZE = 224
RESIZE_SIZE = 256               # eval: resize shorter side to 256, then center-crop 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

MODEL_NAME = "mobilenet_v3_large"   # or "efficientnet_b0"
BATCH_SIZE = 64
EPOCHS_A = 8
EPOCHS_B = 8
LR_A = 1e-3
LR_B = 3e-4
EPOCH_SAMPLES_B = 24_000        # samples drawn per Stage B epoch
SOURCE_MIX_B = {"plantvillage": 0.5, "plantdoc": 0.3, "rice": 0.2}  # share of each source per Stage B epoch
RUN_AUG_ABLATION = False        # True: also train "lab data + field augmentation" to separate the two effects

PV_SPLIT = (0.80, 0.10, 0.10)   # train / val / test
RICE_SPLIT = (0.70, 0.15, 0.15)
PLANTDOC_VAL_FRACTION = 0.15    # taken from the official PlantDoc train set; the official test set stays the field test
NEAR_DUP_HAMMING = 4            # dHash distance (out of 64 bits) treated as "same photo"

USE_DRIVE = True                # save checkpoints + results to Google Drive so a disconnect does not lose the run
DRIVE_ROOT = Path("/content/drive/MyDrive/plant-identifier")

random.seed(SEED)
np.random.seed(SEED)
for d in (RAW_DIR, CACHE_DIR, ARTIFACT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Checkpoints go to Google Drive (Colab asks for permission once). Re-running the notebook after a
# disconnect resumes from the last finished epoch instead of starting over.
PERSIST_ROOT = WORK_DIR / "persist"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PERSIST_ROOT = DRIVE_ROOT
    except Exception as exc:
        print("Google Drive not available; checkpoints stay on this runtime only:", exc)
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
print("Config OK. Persistent storage:", PERSIST_ROOT)

## 2. Download datasets
- PlantVillage: only `raw/color` is fetched (sparse checkout), not the grayscale/segmented copies.
- PlantDoc: classification split (`train/`, `test/`) from the official repository.
- Rice: Kaggle API, using the credentials stored in Colab secrets.

In [ ]:
def run(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

if not PV_COLOR_DIR.exists():
    run(f"git clone --depth 1 --filter=blob:none --sparse "
        f"https://github.com/spMohanty/PlantVillage-Dataset {PV_REPO_DIR}")
    run("git sparse-checkout set raw/color", cwd=PV_REPO_DIR)

if not (PLANTDOC_DIR / "train").exists():
    run(f"git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset {PLANTDOC_DIR}")

def kaggle_credentials_available():
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        return True
    except Exception:
        return False

rice_ready = RICE_DIR.exists() and any(p.suffix.lower() in {".jpg", ".jpeg", ".png"} for p in RICE_DIR.rglob("*"))
if INCLUDE_RICE and not rice_ready:
    if kaggle_credentials_available():
        RICE_DIR.mkdir(parents=True, exist_ok=True)
        try:
            run(f"kaggle datasets download -d {RICE_KAGGLE_SLUG} -p {RICE_DIR} --unzip")
        except subprocess.CalledProcessError:
            print("WARNING: rice download failed; continuing without rice.")
            INCLUDE_RICE = False
    else:
        print("WARNING: Kaggle secrets KAGGLE_USERNAME / KAGGLE_KEY not found; continuing without rice.\n"
              "Add them (key icon in the sidebar, enable notebook access) and re-run to include rice.")
        INCLUDE_RICE = False

print("Downloads ready. Rice included:", INCLUDE_RICE)

## 3. Label mapping
Every class is named `Plant___Condition`, using clean English ids that the API and the Android app
will use as keys (`GET /plants/{id}`). PlantDoc and rice folders are mapped onto the same names.
Any folder that cannot be mapped stops the notebook with a clear message instead of being silently dropped.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def norm(name):
    """Normalize a folder name: lowercase, any non-alphanumeric run -> single space."""
    return re.sub(r"[^a-z0-9]+", " ", name.lower()).strip()

def list_images(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

PLANTVILLAGE_MAP = {norm(k): v for k, v in {
    "Apple___Apple_scab": "Apple___Scab",
    "Apple___Black_rot": "Apple___Black_rot",
    "Apple___Cedar_apple_rust": "Apple___Cedar_apple_rust",
    "Apple___healthy": "Apple___Healthy",
    "Blueberry___healthy": "Blueberry___Healthy",
    "Cherry_(including_sour)___Powdery_mildew": "Cherry___Powdery_mildew",
    "Cherry_(including_sour)___healthy": "Cherry___Healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": "Corn___Gray_leaf_spot",
    "Corn_(maize)___Common_rust_": "Corn___Common_rust",
    "Corn_(maize)___Northern_Leaf_Blight": "Corn___Northern_leaf_blight",
    "Corn_(maize)___healthy": "Corn___Healthy",
    "Grape___Black_rot": "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)": "Grape___Esca",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)": "Grape___Leaf_blight",
    "Grape___healthy": "Grape___Healthy",
    "Orange___Haunglongbing_(Citrus_greening)": "Orange___Citrus_greening",
    "Peach___Bacterial_spot": "Peach___Bacterial_spot",
    "Peach___healthy": "Peach___Healthy",
    "Pepper,_bell___Bacterial_spot": "Pepper_bell___Bacterial_spot",
    "Pepper,_bell___healthy": "Pepper_bell___Healthy",
    "Potato___Early_blight": "Potato___Early_blight",
    "Potato___Late_blight": "Potato___Late_blight",
    "Potato___healthy": "Potato___Healthy",
    "Raspberry___healthy": "Raspberry___Healthy",
    "Soybean___healthy": "Soybean___Healthy",
    "Squash___Powdery_mildew": "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch": "Strawberry___Leaf_scorch",
    "Strawberry___healthy": "Strawberry___Healthy",
    "Tomato___Bacterial_spot": "Tomato___Bacterial_spot",
    "Tomato___Early_blight": "Tomato___Early_blight",
    "Tomato___Late_blight": "Tomato___Late_blight",
    "Tomato___Leaf_Mold": "Tomato___Leaf_mold",
    "Tomato___Septoria_leaf_spot": "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite": "Tomato___Spider_mites",
    "Tomato___Target_Spot": "Tomato___Target_spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": "Tomato___Yellow_leaf_curl_virus",
    "Tomato___Tomato_mosaic_virus": "Tomato___Mosaic_virus",
    "Tomato___healthy": "Tomato___Healthy",
}.items()}

# PlantDoc folder names -> PlantVillage-compatible labels.
# "<Plant> leaf" folders are healthy leaves. "Corn leaf blight" is treated as Northern leaf blight.
PLANTDOC_MAP = {norm(k): v for k, v in {
    "Apple Scab Leaf": "Apple___Scab",
    "Apple leaf": "Apple___Healthy",
    "Apple rust leaf": "Apple___Cedar_apple_rust",
    "Bell_pepper leaf": "Pepper_bell___Healthy",
    "Bell_pepper leaf spot": "Pepper_bell___Bacterial_spot",
    "Blueberry leaf": "Blueberry___Healthy",
    "Cherry leaf": "Cherry___Healthy",
    "Corn Gray leaf spot": "Corn___Gray_leaf_spot",
    "Corn leaf blight": "Corn___Northern_leaf_blight",
    "Corn rust leaf": "Corn___Common_rust",
    "Peach leaf": "Peach___Healthy",
    "Potato leaf": "Potato___Healthy",
    "Potato leaf early blight": "Potato___Early_blight",
    "Potato leaf late blight": "Potato___Late_blight",
    "Raspberry leaf": "Raspberry___Healthy",
    "Soyabean leaf": "Soybean___Healthy",
    "Soybean leaf": "Soybean___Healthy",
    "Squash Powdery mildew leaf": "Squash___Powdery_mildew",
    "Strawberry leaf": "Strawberry___Healthy",
    "Tomato Early blight leaf": "Tomato___Early_blight",
    "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato leaf": "Tomato___Healthy",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato leaf late blight": "Tomato___Late_blight",
    "Tomato leaf mosaic virus": "Tomato___Mosaic_virus",
    "Tomato leaf yellow virus": "Tomato___Yellow_leaf_curl_virus",
    "Tomato mold leaf": "Tomato___Leaf_mold",
    "Tomato two spotted spider mites leaf": "Tomato___Spider_mites",
    "grape leaf": "Grape___Healthy",
    "grape leaf black rot": "Grape___Black_rot",
}.items()}

def rice_label(folder_name):
    n = norm(folder_name)
    words = n.split()
    if "blast" in n:
        return "Rice___Blast"
    if "bacterial" in n or "blight" in n or "blb" in words:
        return "Rice___Bacterial_leaf_blight"
    if "brown" in n:
        return "Rice___Brown_spot"
    if "healthy" in n or "normal" in words:
        return "Rice___Healthy"
    return None

def collect_plantvillage(color_dir):
    records, unmapped = [], []
    for class_dir in sorted(p for p in Path(color_dir).iterdir() if p.is_dir()):
        label = PLANTVILLAGE_MAP.get(norm(class_dir.name))
        if label is None:
            unmapped.append(class_dir.name)
            continue
        for img in list_images(class_dir):
            records.append({"src_path": str(img), "source": "plantvillage", "label": label, "orig_split": ""})
    if unmapped:
        raise ValueError(f"Unmapped PlantVillage folders (add them to PLANTVILLAGE_MAP): {unmapped}")
    return records

def collect_plantdoc(root):
    records, unmapped = [], []
    for split in ("train", "test"):
        for class_dir in sorted(p for p in (Path(root) / split).iterdir() if p.is_dir()):
            label = PLANTDOC_MAP.get(norm(class_dir.name))
            if label is None:
                unmapped.append(f"{split}/{class_dir.name}")
                continue
            for img in list_images(class_dir):
                records.append({"src_path": str(img), "source": "plantdoc", "label": label, "orig_split": split})
    if unmapped:
        raise ValueError(f"Unmapped PlantDoc folders (add them to PLANTDOC_MAP): {unmapped}")
    return records

AUG_TOKENS = {"orig", "original", "rotated", "rotate", "rot", "flip", "flipped", "mirror", "mirrored",
              "aug", "augmented", "copy", "img", "image", "jpg", "jpeg", "png"}

def rice_group_key(path):
    """'blast_rotated_009.jpg' and 'blast_orig_009.jpg' -> same key, so augmented copies share a split."""
    tokens = [t for t in norm(Path(path).stem).split() if t not in AUG_TOKENS]
    return "rice:" + " ".join(tokens)

def rice_label_from_annotation(img):
    """Supervisely layout: <split>/img/<name> with <split>/ann/<name>.json listing object classTitles."""
    ann = img.parent.parent / "ann" / f"{img.name}.json"
    if not ann.exists():
        return None, False
    titles = {o.get("classTitle", "") for o in json.loads(ann.read_text()).get("objects", [])}
    labels = {rice_label(t) for t in titles} - {None}
    if len(labels) == 1:
        return labels.pop(), True
    return None, True          # annotated but empty / several diseases in one image -> skip

def collect_rice(root):
    """Label each image from its annotation file, else its folder name, else its file name."""
    records, skipped = [], Counter()
    for img in list_images(root):
        label, annotated = rice_label_from_annotation(img)
        if not annotated:
            label = rice_label(img.parent.name) or rice_label(img.stem)
        if label is None:
            skipped["annotated, mixed or empty" if annotated else img.parent.name] += 1
            continue
        records.append({"src_path": str(img), "source": "rice", "label": label, "orig_split": "",
                        "group": rice_group_key(img)})
    if skipped:
        print("Rice images skipped:", dict(skipped))
    if not records:
        raise ValueError(f"No rice images mapped under {root}")
    print("Rice images per label:", dict(Counter(r["label"] for r in records)))
    return records

records = collect_plantvillage(PV_COLOR_DIR) + collect_plantdoc(PLANTDOC_DIR)
if INCLUDE_RICE:
    records += collect_rice(RICE_DIR)

raw_df = pd.DataFrame(records)
pv_labels = set(raw_df.loc[raw_df.source == "plantvillage", "label"])
if len(pv_labels) != 38:
    print(f"WARNING: expected 38 PlantVillage classes, found {len(pv_labels)}")
pd_only = set(raw_df.loc[raw_df.source == "plantdoc", "label"]) - pv_labels
assert not pd_only, f"PlantDoc labels missing from PlantVillage: {pd_only}"

print(raw_df.groupby("source").agg(images=("label", "size"), classes=("label", "nunique")))

## 4. Cache resized images and compute hashes
Each image is decoded once, EXIF-rotated, resized (shorter side 320 px) and saved as JPEG.
At the same time we compute an MD5 of the original bytes (exact duplicates) and a 64-bit
dHash (near duplicates: same photo re-encoded, resized or lightly edited).

In [ ]:
INDEX_CSV = WORK_DIR / "index.csv"
INDEX_META = WORK_DIR / "index_meta.json"

def dhash_hex(img):
    gray = np.asarray(img.convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    bits = (gray[:, 1:] > gray[:, :-1]).flatten()
    value = 0
    for bit in bits:
        value = (value << 1) | int(bit)
    return f"{value:016x}"

def process_one(rec):
    src = Path(rec["src_path"])
    try:
        data = src.read_bytes()
        img = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert("RGB")
    except Exception:
        return None
    md5 = hashlib.md5(data).hexdigest()
    w, h = img.size
    scale = CACHE_SHORT_SIDE / min(w, h)
    if scale < 1:
        img = img.resize((max(1, round(w * scale)), max(1, round(h * scale))), Image.BICUBIC)
    out = CACHE_DIR / rec["source"] / f"{md5}.jpg"
    if not out.exists():
        img.save(out, "JPEG", quality=95)
    return {**rec, "path": str(out), "md5": md5, "dhash": dhash_hex(img), "width": w, "height": h}

meta = {"n_records": len(records), "cache_short_side": CACHE_SHORT_SIDE}
if INDEX_CSV.exists() and INDEX_META.exists() and json.loads(INDEX_META.read_text()) == meta:
    df_all = pd.read_csv(INDEX_CSV, dtype=str, keep_default_na=False)
    print("Loaded cached index")
else:
    for source in raw_df.source.unique():
        (CACHE_DIR / source).mkdir(parents=True, exist_ok=True)
    with Pool(max(1, os.cpu_count() or 1)) as pool:
        results = list(tqdm(pool.imap(process_one, records, chunksize=64), total=len(records), desc="Caching"))
    unreadable = [r["src_path"] for r, out in zip(records, results) if out is None]
    if unreadable:
        print(f"Skipped {len(unreadable)} unreadable images, e.g. {unreadable[:3]}")
    df_all = pd.DataFrame([r for r in results if r is not None]).astype(str)
    df_all.to_csv(INDEX_CSV, index=False)
    INDEX_META.write_text(json.dumps(meta))
print(f"{len(df_all)} images indexed")

## 5. Leak-free train / val / test split
1. Exact duplicates (same MD5): copies with **conflicting labels** are removed entirely;
   otherwise one copy is kept (the PlantDoc test copy wins).
2. Images with the same dHash form a group (for rice, also `*_orig_*` / `*_rotated_*` copies of the
   same photo, which dHash cannot match); a group always lands in a single split.
   PlantVillage and rice are split per class by group. PlantDoc keeps its **official test set**
   as the field test; its official train set is split into train/val.
3. Any training image within `NEAR_DUP_HAMMING` bits of a val/test image is removed from training
   (PlantDoc is known to have overlap between its train and test sets).
4. Final assertions: no MD5 or dHash is shared between splits.

In [ ]:
POPCOUNT8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def to_u64(hashes):
    return np.array([int(h, 16) for h in hashes], dtype=np.uint64)

def near_dup_mask(query, ref, max_dist, chunk=64):
    """Boolean mask over `ref`: True where a ref hash is within `max_dist` bits of any query hash."""
    hit = np.zeros(len(ref), dtype=bool)
    if len(query) == 0 or len(ref) == 0:
        return hit
    for i in range(0, len(query), chunk):
        x = np.bitwise_xor(ref[None, :], query[i:i + chunk, None])
        dist = POPCOUNT8[x.view(np.uint8)].reshape(x.shape[0], x.shape[1], 8).sum(axis=-1)
        hit |= (dist <= max_dist).any(axis=0)
    return hit

def group_split(frame, fracs, seed):
    """Assign each group to train/val/test, stratified by the group's majority label."""
    rng = np.random.default_rng(seed)
    group_label = frame.groupby("group")["label"].agg(lambda s: s.value_counts().index[0])
    assignment = {}
    for label in sorted(group_label.unique()):
        keys = np.array(sorted(group_label.index[group_label == label]))
        rng.shuffle(keys)
        n = len(keys)
        n_test = int(round(n * fracs[2])) if n >= 3 else 0
        n_val = int(round(n * fracs[1])) if n >= 3 else 0
        n_val = max(0, min(n_val, n - n_test - 1))
        for i, key in enumerate(keys):
            assignment[key] = "test" if i < n_test else ("val" if i < n_test + n_val else "train")
    return frame["group"].map(assignment)

df = df_all.copy()
log = {}

# group id: rice uses its file-name key (orig/rotated copies), everything else its dHash
if "group" not in df.columns:
    df["group"] = ""
df["group"] = df["group"].fillna("").replace("nan", "")
df.loc[df.group == "", "group"] = "h:" + df.loc[df.group == "", "dhash"]

# 1) exact duplicates
n_labels = df.groupby("md5")["label"].transform("nunique")
log["exact_dup_label_conflict_removed"] = int((n_labels > 1).sum())
df = df[n_labels == 1].copy()
df["_prio"] = (df["orig_split"] != "test").astype(int)
before = len(df)
df = df.sort_values(["_prio", "source", "src_path"]).drop_duplicates("md5", keep="first").drop(columns="_prio")
log["exact_dup_copies_removed"] = before - len(df)

# 2) group-aware split
df["split"] = ""
is_pv = df.source == "plantvillage"
df.loc[is_pv, "split"] = group_split(df[is_pv], PV_SPLIT, SEED)
is_rice = df.source == "rice"
if is_rice.any():
    df.loc[is_rice, "split"] = group_split(df[is_rice], RICE_SPLIT, SEED)
is_pd_test = (df.source == "plantdoc") & (df.orig_split == "test")
is_pd_train = (df.source == "plantdoc") & (df.orig_split == "train")
df.loc[is_pd_test, "split"] = "test"
df.loc[is_pd_train, "split"] = group_split(
    df[is_pd_train], (1 - PLANTDOC_VAL_FRACTION, PLANTDOC_VAL_FRACTION, 0.0), SEED)
assert (df.split != "").all()

# same dHash in several splits (e.g. PlantDoc train vs official test): keep test > val > train
rank = df.split.map({"test": 0, "val": 1, "train": 2})
leak = rank > rank.groupby(df.dhash).transform("min")
log["same_dhash_removed"] = int(leak.sum())
df = df[~leak].copy()

# 3) near duplicates: drop non-test images close to a test image, then train images close to a val image
for eval_split, ref_splits in (("test", ["train", "val"]), ("val", ["train"])):
    query = to_u64(df.loc[df.split == eval_split, "dhash"])
    ref_idx = df.index[df.split.isin(ref_splits)]
    mask = near_dup_mask(query, to_u64(df.loc[ref_idx, "dhash"]), NEAR_DUP_HAMMING)
    log[f"near_dup_of_{eval_split}_removed"] = int(mask.sum())
    df = df.drop(index=ref_idx[mask])

# 4) assertions
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    for col in ("md5", "dhash"):
        shared = set(df.loc[df.split == a, col]) & set(df.loc[df.split == b, col])
        assert not shared, f"{len(shared)} {col} values shared between {a} and {b}"

df = df.sort_values(["source", "split", "label", "md5"]).reset_index(drop=True)
CLASSES = sorted(df.label.unique())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
# A run is identified by its classes, split and training settings: a resumed session only reuses
# checkpoints that were produced with exactly the same setup.
_run_key = json.dumps({"classes": CLASSES, "model": MODEL_NAME, "epochs": [EPOCHS_A, EPOCHS_B],
                       "lr": [LR_A, LR_B], "seed": SEED, "train_md5": sorted(df.loc[df.split == "train", "md5"])},
                      sort_keys=True)
RUN_ID = hashlib.sha1(_run_key.encode()).hexdigest()[:10]
PERSIST_DIR = PERSIST_ROOT / f"run-{RUN_ID}"
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
PLANTS = sorted({c.split("___")[0] for c in CLASSES})
PLANT_TO_IDX = {p: i for i, p in enumerate(PLANTS)}
CLASS_PLANT_IDX = np.array([PLANT_TO_IDX[c.split("___")[0]] for c in CLASSES])
PLANT_ONEHOT = np.eye(len(PLANTS))[CLASS_PLANT_IDX]          # (num_classes, num_plants)
PLANT_CONDITIONS = {p: sorted(c.split("___")[1] for c in CLASSES if c.startswith(p + "___")) for p in PLANTS}
# A leaf-condition diagnosis is only meaningful when the plant has a Healthy class AND at least one disease.
# Otherwise (e.g. Orange: only Citrus_greening, Soybean: only Healthy) the model can only ever give one answer,
# so the app shows the plant and its care info, and says the condition cannot be assessed.
CONDITION_SUPPORTED = {p: "Healthy" in conds and len(conds) >= 2 for p, conds in PLANT_CONDITIONS.items()}
train_df = df[df.split == "train"].reset_index(drop=True)
val_df = df[df.split == "val"].reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)

print("Dedup log:", json.dumps(log, indent=2))
print(f"{len(CLASSES)} classes, {len(PLANTS)} plants")
print("Run id:", RUN_ID, "| checkpoints:", PERSIST_DIR)
print("Condition diagnosis NOT supported for:", [p for p, ok in CONDITION_SUPPORTED.items() if not ok])
print(pd.crosstab(df.source, df.split, margins=True))

manifest = df.assign(src_path=df.src_path.str.replace(str(RAW_DIR) + "/", "", regex=False))
manifest[["source", "src_path", "label", "split", "group", "md5", "dhash"]].to_csv(ARTIFACT_DIR / "split_manifest.csv", index=False)
(ARTIFACT_DIR / "dedup_log.json").write_text(json.dumps(log, indent=2))

In [ ]:
import matplotlib.pyplot as plt

def show_samples(frame, n_per_source=6):
    sources = sorted(frame.source.unique())
    fig, axes = plt.subplots(len(sources), n_per_source, figsize=(2.2 * n_per_source, 2.4 * len(sources)))
    axes = np.atleast_2d(axes)
    for r, source in enumerate(sources):
        sample = frame[frame.source == source].sample(n_per_source, random_state=SEED)
        for c, (_, row) in enumerate(sample.iterrows()):
            ax = axes[r, c]
            ax.imshow(Image.open(row.path))
            ax.set_title(f"{source}\n{row.label}", fontsize=7)
            ax.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(train_df)

## 6. Data pipeline and model
- **Lab augmentation** (Stage A): mild crop, flip and color jitter — a typical "train on PlantVillage" setup.
- **Field augmentation** (Stage B): aggressive crops (leaf partly visible), rotation, perspective,
  strong lighting/color changes, blur, JPEG artifacts and random occlusion.

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms as T

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
device = "cuda"
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
NUM_WORKERS = max(2, os.cpu_count() or 2)
print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))

class RandomJPEG:
    """Re-encode as JPEG with random quality to mimic phone camera / messaging compression."""
    def __init__(self, quality=(30, 90), p=0.3):
        self.quality, self.p = quality, p

    def __call__(self, img):
        if random.random() >= self.p:
            return img
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=random.randint(*self.quality))
        buf.seek(0)
        return Image.open(buf).convert("RGB")

lab_train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.1, 0.1, 0.1),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

field_train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.25, 1.0), ratio=(0.75, 1.33)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation(30)], p=0.5),
    T.RandomPerspective(distortion_scale=0.3, p=0.3),
    T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.05)], p=0.8),
    T.RandomGrayscale(p=0.05),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3),
    RandomJPEG(p=0.3),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

eval_tf = T.Compose([
    T.Resize(RESIZE_SIZE),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

class LeafDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame.path.tolist()
        self.labels = [CLASS_TO_IDX[label] for label in frame.label]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]

def make_loader(frame, transform, sampler=None):
    return DataLoader(
        LeafDataset(frame, transform), batch_size=BATCH_SIZE, shuffle=sampler is None, sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, persistent_workers=True)

def build_model(num_classes):
    if MODEL_NAME == "mobilenet_v3_large":
        model = torchvision.models.mobilenet_v3_large(
            weights=torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    elif MODEL_NAME == "efficientnet_b0":
        model = torchvision.models.efficientnet_b0(
            weights=torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Unknown MODEL_NAME: {MODEL_NAME}")
    return model.to(device)

@torch.no_grad()
def predict_probs(model, frame, batch_size=128):
    loader = DataLoader(LeafDataset(frame, eval_tf), batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    model.eval()
    out = []
    for x, _ in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x.to(device, non_blocking=True))
        out.append(logits.float().softmax(dim=1).cpu())
    return torch.cat(out).numpy()

def accuracy_by_source(model, frame):
    result = {}
    for source, part in frame.groupby("source"):
        y = part.label.map(CLASS_TO_IDX).to_numpy()
        result[source] = float((predict_probs(model, part).argmax(1) == y).mean())
    return result

def _atomic_save(obj, path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def train_stage(tag, model, loader, epochs, lr, select_sources):
    """Train with AdamW + warmup/cosine; keep the epoch with the best mean val accuracy on `select_sources`.
    State is saved to PERSIST_DIR after every epoch, so re-running after a disconnect resumes."""
    best_path = PERSIST_DIR / f"{tag}_best.pt"
    last_path = PERSIST_DIR / f"{tag}_last.pt"
    done_path = PERSIST_DIR / f"{tag}_done.json"
    if done_path.exists() and best_path.exists():
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"[{tag}] finished in an earlier session: loaded the best checkpoint from {PERSIST_DIR}")
        return model, json.loads(done_path.read_text())["history"]

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    total_steps = epochs * len(loader)
    warmup = max(1, min(500, total_steps // 10))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warmup if s < warmup
        else 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total_steps - warmup))))
    scaler = torch.amp.GradScaler("cuda")
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    start_epoch, best, history = 1, -1.0, []
    if last_path.exists():
        state = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(state["model"])
        opt.load_state_dict(state["opt"])
        sched.load_state_dict(state["sched"])
        scaler.load_state_dict(state["scaler"])
        start_epoch, best, history = state["epoch"] + 1, state["best"], state["history"]
        print(f"[{tag}] resuming after epoch {state['epoch']} (best score so far {best:.4f})")

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        t0, loss_sum, seen = time.time(), 0.0, 0
        for x, y in tqdm(loader, desc=f"{tag} {epoch}/{epochs}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            loss_sum += loss.item() * len(y)
            seen += len(y)
        val_acc = accuracy_by_source(model, val_df)
        score = float(np.mean([val_acc[s] for s in select_sources if s in val_acc]))
        history.append({"epoch": epoch, "train_loss": loss_sum / seen, "val_acc": val_acc, "score": score})
        marker = ""
        if score > best:
            best, marker = score, "  <- best"
            _atomic_save(model.state_dict(), best_path)
        _atomic_save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                      "scaler": scaler.state_dict(), "epoch": epoch, "best": best, "history": history}, last_path)
        accs = "  ".join(f"{k}={v:.3f}" for k, v in val_acc.items())
        print(f"[{tag}] epoch {epoch:2d}  loss={loss_sum / seen:.4f}  val: {accs}  ({time.time() - t0:.0f}s){marker}")

    done_path.write_text(json.dumps({"history": history, "best": best}))
    last_path.unlink(missing_ok=True)   # the large optimizer state is no longer needed
    model.load_state_dict(torch.load(best_path, map_location=device))
    return model, history

## 7. Stage A — lab-only baseline (PlantVillage)
Checkpoint selection uses **PlantVillage val only**, as a lab-only project would.
PlantDoc val accuracy is printed for information, not used for selection.

In [ ]:
histories = {}
TRAINED_SOURCES = {}

model_A = build_model(len(CLASSES))
model_A, histories["A_lab_only"] = train_stage(
    "A_lab_only", model_A, make_loader(train_df[train_df.source == "plantvillage"], lab_train_tf),
    EPOCHS_A, LR_A, select_sources=["plantvillage"])
TRAINED_SOURCES["A_lab_only"] = ["plantvillage"]

In [ ]:
if RUN_AUG_ABLATION:
    model_A_aug = build_model(len(CLASSES))
    model_A_aug, histories["A_lab_field_aug"] = train_stage(
        "A_lab_field_aug", model_A_aug, make_loader(train_df[train_df.source == "plantvillage"], field_train_tf),
        EPOCHS_A, LR_A, select_sources=["plantvillage"])
    TRAINED_SOURCES["A_lab_field_aug"] = ["plantvillage"]
else:
    print("Ablation skipped (RUN_AUG_ABLATION = False)")

## 8. Stage B — fine-tune with field data
Starts from Stage A weights. Each epoch samples sources according to `SOURCE_MIX_B`
so the small field datasets are not drowned out by PlantVillage.
Checkpoint selection uses the mean val accuracy over all sources.

In [ ]:
counts = train_df.source.value_counts().to_dict()
mix_total = sum(SOURCE_MIX_B[s] for s in counts)
weights = train_df.source.map(lambda s: SOURCE_MIX_B[s] / mix_total / counts[s]).to_numpy()
sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=EPOCH_SAMPLES_B,
                                replacement=True, generator=torch.Generator().manual_seed(SEED))
print("Expected share per epoch:", {s: round(SOURCE_MIX_B[s] / mix_total, 2) for s in counts})

model_B = build_model(len(CLASSES))
model_B.load_state_dict(model_A.state_dict())
model_B, histories["B_field_finetuned"] = train_stage(
    "B_field_finetuned", model_B, make_loader(train_df, field_train_tf, sampler=sampler),
    EPOCHS_B, LR_B, select_sources=sorted(counts))
TRAINED_SOURCES["B_field_finetuned"] = sorted(counts)

(ARTIFACT_DIR / "training_history.json").write_text(json.dumps(histories, indent=2))

## 9. Evaluation: lab test vs field test
For every model and test set: accuracy, top-3 accuracy (the API returns top-3), macro-F1,
plant-level accuracy (correct crop, ignoring the disease), per-class report and a row-normalized
confusion matrix. Predictions of a class that is absent from that test set go to an "other" column.

In [ ]:
from sklearn.metrics import classification_report, f1_score

EVAL_DIR = ARTIFACT_DIR / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_TITLES = {"plantvillage": "PlantVillage test (lab)", "plantdoc": "PlantDoc test (field)", "rice": "Rice test"}

def plant_of(label):
    return label.split("___")[0]

def plot_confusion(y, pred, present, names_all, title, path):
    """Row-normalized confusion matrix; predictions outside `present` go to an '(other)' column."""
    pos = {c: i for i, c in enumerate(present)}
    cm = np.zeros((len(present), len(present) + 1))
    for t, p in zip(y, pred):
        cm[pos[t], pos.get(p, len(present))] += 1
    cm = cm / cm.sum(axis=1, keepdims=True)
    names = [names_all[i] for i in present]
    size = max(6, 0.38 * len(present) + 3)
    fig, ax = plt.subplots(figsize=(size + 1, size))
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(present) + 1), names + ["(other)"], rotation=90, fontsize=7)
    ax.set_yticks(range(len(present)), names, fontsize=7)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    fig.colorbar(im, fraction=0.03)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)

def evaluate(tag, model):
    results = {}
    sources = ["plantvillage", "plantdoc"] + [s for s in ["rice"] if s in TRAINED_SOURCES[tag]]
    for source in sources:
        part = test_df[test_df.source == source]
        if part.empty:
            continue
        probs = predict_probs(model, part)
        y = part.label.map(CLASS_TO_IDX).to_numpy()
        pred = probs.argmax(1)
        top3 = np.argsort(-probs, axis=1)[:, :3]
        present = sorted(np.unique(y).tolist())
        report = classification_report(y, pred, labels=present, target_names=[CLASSES[i] for i in present],
                                       output_dict=True, zero_division=0)
        pd.DataFrame(report).T.to_csv(EVAL_DIR / f"{tag}__{source}__per_class.csv")
        plot_confusion(y, pred, present, CLASSES, f"{tag} — {SOURCE_TITLES[source]} (disease level)",
                       EVAL_DIR / f"{tag}__{source}__confusion.png")

        # plant level: sum class probabilities per plant (what the app shows as "this is a ...")
        plant_probs = probs @ PLANT_ONEHOT
        y_plant = CLASS_PLANT_IDX[y]
        pred_plant = plant_probs.argmax(1)
        plant_present = sorted(np.unique(y_plant).tolist())
        plant_report = classification_report(y_plant, pred_plant, labels=plant_present,
                                             target_names=[PLANTS[i] for i in plant_present],
                                             output_dict=True, zero_division=0)
        pd.DataFrame(plant_report).T.to_csv(EVAL_DIR / f"{tag}__{source}__per_plant.csv")
        plot_confusion(y_plant, pred_plant, plant_present, PLANTS, f"{tag} — {SOURCE_TITLES[source]} (plant level)",
                       EVAL_DIR / f"{tag}__{source}__confusion_plant.png")

        supported = np.array([CONDITION_SUPPORTED[PLANTS[i]] for i in y_plant])
        results[source] = {
            "n_images": int(len(y)),
            "n_classes": len(present),
            "accuracy": float((pred == y).mean()),
            "top3_accuracy": float((top3 == y[:, None]).any(axis=1).mean()),
            "macro_f1": float(f1_score(y, pred, labels=present, average="macro", zero_division=0)),
            "plant_accuracy": float((pred_plant == y_plant).mean()),
            "plant_macro_f1": float(f1_score(y_plant, pred_plant, labels=plant_present, average="macro", zero_division=0)),
            "condition_accuracy_supported_plants": float((pred == y)[supported].mean()) if supported.any() else None,
        }
    return results

models = {"A_lab_only": model_A, "B_field_finetuned": model_B}
if RUN_AUG_ABLATION:
    models = {"A_lab_only": model_A, "A_lab_field_aug": model_A_aug, "B_field_finetuned": model_B}
all_results = {tag: evaluate(tag, m) for tag, m in models.items()}
(ARTIFACT_DIR / "results.json").write_text(json.dumps(all_results, indent=2))

# Markdown tables for the README
def pct(v):
    return "—" if v is None else f"{v:.1%}"

lines = ["**Plant level** — what the app shows as \"this is a ...\"", "",
         "| Model | Test set | Images | Plant accuracy | Plant macro-F1 |",
         "|---|---|---:|---:|---:|"]
for tag, res in all_results.items():
    for source, m in res.items():
        lines.append(f"| {tag} | {SOURCE_TITLES[source]} | {m['n_images']} | {pct(m['plant_accuracy'])} | {m['plant_macro_f1']:.3f} |")
lines += ["", "**Disease level** (Plant___Condition)", "",
          "| Model | Test set | Classes | Accuracy | Top-3 | Macro-F1 | Accuracy on condition-supported plants |",
          "|---|---|---:|---:|---:|---:|---:|"]
for tag, res in all_results.items():
    for source, m in res.items():
        lines.append(f"| {tag} | {SOURCE_TITLES[source]} | {m['n_classes']} | {pct(m['accuracy'])} | "
                     f"{pct(m['top3_accuracy'])} | {m['macro_f1']:.3f} | {pct(m['condition_accuracy_supported_plants'])} |")
results_md = "\n".join(lines)
(ARTIFACT_DIR / "results.md").write_text(results_md + "\n")
print(results_md)

In [ ]:
# Per-class F1 on the field test set: before vs after field fine-tuning
before = pd.read_csv(EVAL_DIR / "A_lab_only__plantdoc__per_class.csv", index_col=0)
after = pd.read_csv(EVAL_DIR / "B_field_finetuned__plantdoc__per_class.csv", index_col=0)
class_rows = [c for c in after.index if c in CLASS_TO_IDX]
field_f1 = pd.DataFrame({
    "support": after.loc[class_rows, "support"].astype(int),
    "f1_before": before.loc[class_rows, "f1-score"].round(3),
    "f1_after": after.loc[class_rows, "f1-score"].round(3),
})
field_f1["delta"] = (field_f1.f1_after - field_f1.f1_before).round(3)
field_f1 = field_f1.sort_values("delta", ascending=False)
field_f1.to_csv(EVAL_DIR / "plantdoc_per_class_f1_before_after.csv")
display(field_f1)

# Training curves
fig, axes = plt.subplots(1, len(histories), figsize=(6 * len(histories), 4), squeeze=False)
for ax, (tag, hist) in zip(axes[0], histories.items()):
    for source in hist[0]["val_acc"]:
        ax.plot([h["epoch"] for h in hist], [h["val_acc"][source] for h in hist], marker="o", label=source)
    ax.set_title(f"{tag}: val accuracy")
    ax.set_xlabel("epoch")
    ax.set_ylim(0, 1)
    ax.legend()
fig.tight_layout()
fig.savefig(EVAL_DIR / "training_curves.png", dpi=130)
plt.show()

### Confidence threshold for "not recognized"
The API names a plant only when its probability is at least `PLANT_MIN_CONFIDENCE`. This table is computed
on the **validation** photos (never the test set) for Model B: a higher threshold answers fewer photos, but is
wrong less often on the photos it does answer. Use it to choose the threshold in `deploy/k8s/configmap.yaml`.

In [ ]:
cal_rows = []
for source, part in val_df.groupby("source"):
    probs = predict_probs(model_B, part)
    plant_probs = probs @ PLANT_ONEHOT
    y_plant = CLASS_PLANT_IDX[part.label.map(CLASS_TO_IDX).to_numpy()]
    conf = plant_probs.max(axis=1)
    correct = plant_probs.argmax(axis=1) == y_plant
    for t in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
        answered = conf >= t
        cal_rows.append({
            "source": source,
            "threshold": t,
            "answered": float(answered.mean()),
            "accuracy_when_answered": float(correct[answered].mean()) if answered.any() else float("nan"),
            "wrong_answers": float((answered & ~correct).mean()),
        })
calibration = pd.DataFrame(cal_rows)
calibration.to_csv(EVAL_DIR / "plant_threshold_calibration.csv", index=False)
display(calibration.pivot(index="threshold", columns="source",
                          values=["answered", "accuracy_when_answered", "wrong_answers"]).round(3))

## 10. Export to ONNX and verify against PyTorch

In [ ]:
import onnx
import onnxruntime as ort

ONNX_PATH = ARTIFACT_DIR / "plant_classifier.onnx"
export_model = copy.deepcopy(model_B).float().cpu().eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
export_kwargs = dict(input_names=["input"], output_names=["logits"], opset_version=17,
                     dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}})
try:
    torch.onnx.export(export_model, (dummy,), str(ONNX_PATH), dynamo=False, **export_kwargs)
except TypeError:   # older torch without the `dynamo` argument
    torch.onnx.export(export_model, (dummy,), str(ONNX_PATH), **export_kwargs)
except Exception as exc:
    print("TorchScript exporter failed, retrying with the dynamo exporter:", exc)
    torch.onnx.export(export_model, (dummy,), str(ONNX_PATH), dynamo=True, **export_kwargs)

# Store everything in a single .onnx file (no external .data file)
onnx_model = onnx.load(str(ONNX_PATH))
onnx.save_model(onnx_model, str(ONNX_PATH), save_as_external_data=False)
for extra in ARTIFACT_DIR.glob("plant_classifier.onnx.*"):
    extra.unlink()
onnx.checker.check_model(str(ONNX_PATH))

session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
check = test_df.groupby("source").head(32)
x = torch.stack([eval_tf(Image.open(p).convert("RGB")) for p in check.path])
with torch.no_grad():
    torch_logits = export_model(x).numpy()
onnx_logits = session.run(["logits"], {"input": x.numpy()})[0]
max_diff = float(np.abs(torch_logits - onnx_logits).max())
top1_agree = float((torch_logits.argmax(1) == onnx_logits.argmax(1)).mean())
print(f"ONNX check on {len(check)} test images: max |logit diff| = {max_diff:.2e}, top-1 agreement = {top1_agree:.1%}")
assert max_diff < 1e-3 and top1_agree == 1.0, "ONNX output does not match PyTorch"

onnx_sha256 = hashlib.sha256(ONNX_PATH.read_bytes()).hexdigest()
print(f"{ONNX_PATH.name}: {ONNX_PATH.stat().st_size / 1e6:.1f} MB, sha256 {onnx_sha256}")

In [ ]:
labels = {
    "model_file": ONNX_PATH.name,
    "model_sha256": onnx_sha256,
    "architecture": MODEL_NAME,
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "num_classes": len(CLASSES),
    "num_plants": len(PLANTS),
    "labels": [{"index": i, "id": c, "plant": plant_of(c), "plant_index": int(CLASS_PLANT_IDX[i]),
                "condition": c.split("___")[1]}
               for i, c in enumerate(CLASSES)],
    "plants": [{"index": i, "id": p, "conditions": PLANT_CONDITIONS[p],
                "condition_supported": CONDITION_SUPPORTED[p]}
               for i, p in enumerate(PLANTS)],
    "inference_rule": ("plant probability = sum of its class probabilities; show the top plant. "
                       "Show the condition only if the plant has condition_supported = true."),
    "input": {"name": "input", "layout": "NCHW", "dtype": "float32", "color": "RGB",
              "resize_shorter_side": RESIZE_SIZE, "center_crop": IMG_SIZE,
              "scale": "pixel / 255", "mean": MEAN, "std": STD},
    "output": {"name": "logits", "note": "apply softmax to get probabilities"},
    "onnx_check": {"max_abs_logit_diff": max_diff, "top1_agreement": top1_agree},
    "license": "CC BY-SA 3.0 (trained on PlantVillage)",
    "datasets": ["PlantVillage (CC BY-SA 3.0)", "PlantDoc (CC BY 4.0)"]
                + ([f"Rice leaf disease, Kaggle {RICE_KAGGLE_SLUG} (CC0 1.0)"] if INCLUDE_RICE else []),
}
(ARTIFACT_DIR / "labels.json").write_text(json.dumps(labels, indent=2))
torch.save(model_B.state_dict(), ARTIFACT_DIR / "plant_classifier_best.pt")
print(json.dumps(labels["labels"][:3], indent=2), "...")

## 11. Package artifacts
The zip contains: `plant_classifier.onnx`, `labels.json`, `results.json`, `results.md`,
`eval/` (confusion matrices, per-class reports, training curves), `split_manifest.csv`,
`dedup_log.json`, `training_history.json` and the PyTorch checkpoint.
A copy of the zip is also saved next to the checkpoints on Google Drive.

In [ ]:
zip_path = shutil.make_archive("/content/plant_classifier_artifacts", "zip", ARTIFACT_DIR)
print("Created", zip_path, f"({Path(zip_path).stat().st_size / 1e6:.1f} MB)")

shutil.copy(zip_path, PERSIST_DIR)
print("Copied to", PERSIST_DIR)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print("Download it from the Files panel instead:", exc)